In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split

import pandas as pd
import numpy as np
from PIL import Image
import os

In [3]:
# CRÉER DATASET PYTORCH

class BrainDataset(torch.utils.data.Dataset):
    def __init__(self, df, base_path, transform):
        self.df = df
        self.base_path = base_path
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.base_path, row["path"])

        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        label = row["label_encoded"]

        return image, label

In [ ]:
#  CHARGEMENT DU CSV

df = pd.read_csv("../outputs/features.csv")

df.head()

,0,1,2,3,4,5,6,7,8,9,...,2040,2041,2042,2043,2044,2045,2046,2047,label,path
0,0.409749,0.873298,0.174183,0.301675,0.067051,0.273842,0.247952,0.647732,0.959357,0.151482,...,0.563092,0.128302,0.720562,0.032226,0.519088,0.430458,1.099062,0.938813,cancer,avec_labels/cancer/05340cd4-3bb2-459d-9937-bf2...
1,0.650230,0.721211,0.054125,0.181203,0.000000,0.292601,0.964571,0.697341,0.577685,0.305469,...,0.185135,0.182850,0.192730,0.043186,0.241807,0.485734,0.671984,0.706141,cancer,avec_labels/cancer/0c6f3641-60d9-4a76-abe5-de8...
2,1.192965,1.131451,0.107712,0.199649,0.000219,0.451692,0.180534,0.693340,0.275049,0.544788,...,0.057330,0.250783,0.273103,0.090985,0.080423,0.438565,0.297450,1.483259,cancer,avec_labels/cancer/0f718241-8f63-4b55-81ce-315...
3,0.494694,0.456065,0.174034,0.257347,0.115510,0.877980,0.727288,0.369569,0.536643,0.290675,...,0.120582,0.462077,0.182437,0.096539,0.537069,0.563836,0.950195,1.112594,cancer,avec_labels/cancer/11a7a426-4806-401e-98b2-b96...
4,0.491905,0.354549,0.390202,0.040721,0.014974,0.305518,0.158039,0.126338,0.265495,0.287022,...,0.830522,0.075254,0.130555,0.143496,0.158412,0.429747,1.463409,1.068511,cancer,avec_labels/cancer/1c043dbb-4623-4769-8e5e-022...


In [7]:
# ENCODAGE LABELS

# labels forts
df_labeled = df[df["label"] != "unlabeled"].copy()

df_labeled["label_encoded"] = df_labeled["label"].map({
    "normal": 0,
    "cancer": 1
})

In [ ]:
#  SPLIT AVANT CLUSTERING

from sklearn.model_selection import train_test_split

df_labeled = df[df["label"] != "unlabeled"].copy()

train_df, test_df = train_test_split(df_labeled, test_size=0.2, random_state=42)

df_unlabeled = df[df["label"] == "unlabeled"].copy()

# clustering uniquement sur train + unlabeled
df_for_clustering = pd.concat([train_df, df_unlabeled])

In [ ]:
# CLUSTERING

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X = df_for_clustering.drop(columns=["label", "path"])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=2, random_state=42)
df_for_clustering["cluster"] = kmeans.fit_predict(X_scaled)

In [ ]:
# MERGE

df = df.merge(
    df_for_clustering[["path", "cluster"]],
    on="path",
    how="left"
)

In [24]:
# MAPPING CLUSTER -> LABEL

# On regarde comment les clusters se répartissent sur les données labellisées
df_labeled_with_cluster = df[df["label"] != "unlabeled"].copy()

mapping = {}

for cluster_id in df["cluster"].unique():
    subset = df_labeled_with_cluster[df_labeled_with_cluster["cluster"] == cluster_id]
    
    if len(subset) > 0:
        majority_label = subset["label"].value_counts().idxmax()
        mapping[cluster_id] = 1 if majority_label == "cancer" else 0

print("Mapping cluster -> label :", mapping)

Mapping cluster -> label : {np.int32(0): 1, np.int32(1): 0}


In [ ]:
# DATA FAIBLEMENT LABELLISÉE

df_unlabeled = df[df["label"] == "unlabeled"].copy()

# pseudo labels venant du clustering
df_unlabeled["label_encoded"] = df_unlabeled["cluster"]

In [ ]:
# TRANSFORMS

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [ ]:
# DATASET

base_path = "../data/raw/mri_dataset_brain_cancer_oc/"

dataset_labeled = BrainDataset(df_labeled, base_path, transform)
dataset_unlabeled = BrainDataset(df_unlabeled, base_path, transform)

In [ ]:
# TRAIN / TEST SPLIT

train_size = int(0.8 * len(dataset_labeled))
test_size = len(dataset_labeled) - train_size

train_dataset, test_dataset = random_split(dataset_labeled, [train_size, test_size])

In [ ]:
# DATALOADER

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

In [ ]:
# MODELE CNN

model = torch.hub.load('pytorch/vision', 'resnet18', pretrained=True)

model.fc = nn.Linear(model.fc.in_features, 2)

Downloading: "https://github.com/pytorch/vision/zipball/main" to C:\Users\selma/.cache\torch\hub\main.zip


c:\Users\selma\Desktop\BrainScanAI\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\selma\Desktop\BrainScanAI\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\selma/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:05<00:00, 9.03MB/s]


In [ ]:
# TRAINING LOOP

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# ENTRAÎNEMENT SUPERVISÉ

for epoch in range(5):
    for images, labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:
# ÉVALUATION

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print("Accuracy supervisé :", accuracy)

Accuracy supervisé : 0.7


In [23]:
print("Accuracy supervisé :", accuracy)

Accuracy supervisé : 0.7


In [ ]:
# SEMI-SUPERVISÉ

combined_df = pd.concat([df_labeled, df_unlabeled])

dataset_combined = BrainDataset(combined_df, base_path, transform)

train_loader_combined = DataLoader(dataset_combined, batch_size=16, shuffle=True)

In [ ]:
# RE-TRAIN

for epoch in range(5):
    for images, labels in train_loader_combined:
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:
# ÉVALUATION FINALE

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy_semi = correct / total

print("Accuracy semi-supervisé :", accuracy_semi)

Accuracy semi-supervisé : 0.6


## Comparaison des performances

- Accuracy supervisé : 0.7
- Accuracy semi-supervisé : 0.6

Le modèle supervisé obtient de meilleures performances.

L'utilisation de pseudo-labels issus du clustering introduit du bruit dans les données,
ce qui dégrade légèrement la performance du modèle.

Cela montre que la qualité des pseudo-labels est un facteur clé en apprentissage semi-supervisé.